# Notebook for computing hashes, buckets and similarity values for the disk scheme using a hybrid approach

Utilizes the disk scheme

Incorporates:
* Hashing of trajectories using disk scheme
* Bucketing of hashes made from disk scheme
* Similarity computation between trajectories within buckets.
    * Both for DTW and Frechet
* Analysis of the produced bucket system

Produces:
* JSON file containing buckets
* Similarity values for trajectories within buckets


## Hybrid approach

In [ ]:
import os
import sys

def find_project_root(target_folder="masteroppgave"):
    """Find the absolute path of a folder by searching upward."""
    currentdir = os.path.abspath("__file__")  # Get absolute script path
    while True:
        if os.path.basename(currentdir) == target_folder:
            return currentdir  # Found the target folder
        parentdir = os.path.dirname(currentdir)
        if parentdir == currentdir:  # Stop at filesystem root
            return None
        currentdir = parentdir  # Move one level up

# Example usage
project_root = find_project_root("masteroppgave")

if project_root:
    sys.path.append(project_root)
    print(f"Project root found: {project_root}")
else:
    raise RuntimeError("Could not find 'masteroppgave' directory")

from computation.similarity import *
from utils.helpers.bucket_evaluation import *
import json
import pandas as pd
from utils.helpers.parallel_computing import grid_gather_bucketing_and_compression_lists, grid_hybrid_compute_evaluation_scores
import numpy as np


# Cell for performing several executions with different parameter values, writes all results to csv

In [ ]:
# Strategies
CITY = "rome" # "rome" or "porto"
MEASURE = "dtw" # "dtw" or "frechet"
BUCKETING_METHOD = "loose" # Bucketing method to use
SIZE = 50  # Example dataset sizes

# Define similarity thresholds
THRESHOLDS = np.arange(1, 2.0, 1)  # Generates [1, 2, 3, 4, 5]

# Dictionary with parameter values
parameter_Values = {
    "config1": {
        "bucketing": {
            "resolution": 1.6,
            "layers": 2,
        },
        "compression": {
            "resolution": 1.5,
            "layers": 5,
        },
        "size": 50 
    },
    "config2": {
        "bucketing": {
            "resolution": 1.8,
            "layers": 2,
        },
        "compression": {
            "resolution": 2,
            "layers": 2,
        },
        "size": 50 
    },
}


(
    compression_resolutions,
    compression_layers,
    bucketing_resolutions,
    bucketing_layers,
) = grid_gather_bucketing_and_compression_lists(parameter_Values)

# Prepare CSV file for results
csv_filename = f"../../../results_hashed/bucket_evaluation/grid_HYBRID_bucket_evaluation_RESOLUTION_b_{bucketing_resolutions}_c_{compression_resolutions}_LAYERS_b_{bucketing_layers}_c_{compression_layers}_results.csv"

# Get true similarity matrix
file_path = f"../../../results_true/similarity_values/{CITY}/{MEASURE}/{CITY}-{MEASURE}-{SIZE}.csv"

print(f"True sim matrix got: {file_path}")

# READ IN TRUE SIM MATRIX
true_sim_matrix_df_unsymetric = pd.read_csv(file_path, index_col=0)

def convert_to_float(value):
    try:
        return float(value)
    except ValueError:
        return value

true_sim_matrix_df = true_sim_matrix_df_unsymetric.map(convert_to_float)
true_sim_matrix_df = (true_sim_matrix_df + true_sim_matrix_df.T)

first_write = True

# Loop through parameter combinations
for config_name, config_data in parameter_Values.items():
    
    # Get the bucketing parameters
    RESOLUTION_BUCKETING = config_data.get("bucketing").get("resolution")
    LAYERS_BUCKETING = config_data.get("bucketing").get("layers")
    
    #Get the compression parameters
    RESOLUTION_COMPRESSION = config_data.get("compression").get("resolution")
    LAYERS_COMPRESSION = config_data.get("compression").get("layers")
    
    #Prints configuration
    print(f"\nRunning hybrid: City={CITY}, Measure={MEASURE}, Resolution_bucketing={RESOLUTION_BUCKETING}, Layers_bucketing={LAYERS_BUCKETING}, Resolution_compression={RESOLUTION_COMPRESSION}, Layers_compression={LAYERS_COMPRESSION}, Size={SIZE}")

    #New configuration
    df = grid_hybrid_compute_evaluation_scores(CITY=CITY, MEASURE=MEASURE, RESOLUTION_BUCKETING=RESOLUTION_BUCKETING, LAYERS_BUCKETING=LAYERS_BUCKETING, RESOLUTION_COMPRESSION=RESOLUTION_COMPRESSION, LAYERS_COMPRESSION=LAYERS_COMPRESSION, SIZE=SIZE, BUCKETING_METHOD=BUCKETING_METHOD, TRUE_SIM_MATRIX=true_sim_matrix_df, TRESHOLDS=THRESHOLDS, iterations=3, parallell_jobs=8)
    df.to_csv(csv_filename, mode='a', header=first_write, index=False)

    # After the first write, disable header writing
    first_write = False
    